In [1]:
from konlpy.tag import Okt
from collections import Counter

content_corpus = '죽는날 까지 하늘을 우러러\
한점 부끄럼이 없기를\
잎새에 이는 바람에도\
나는 괴로워 했다\
별을 노래하는 마음으로\
모든 죽어가는 것을 사랑해야지\
그리고 나에게 주어진 길을\
걸어가야겠다.\
오늘밤에도 별이 바람에 스치운다'

# konlpy의 형태소 분석기로 명사 단위의 키워드를 추출합니다.
nouns_tagger = Okt()
nouns = nouns_tagger.nouns(content_corpus)
count = Counter(nouns)

In [2]:
count

Counter({'바람': 2,
         '별': 2,
         '날': 1,
         '하늘': 1,
         '우러러한점': 1,
         '부끄럼': 1,
         '잎새': 1,
         '도나': 1,
         '노래': 1,
         '마음': 1,
         '모든': 1,
         '것': 1,
         '사랑': 1,
         '나': 1,
         '길': 1,
         '오늘밤': 1})

In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_twitter(keyword, scroll_count=5):
    # 1. 크롬 드라이버 및 옵션 설정
    chrome_options = Options()
    # 필요 시 크롬창을 띄우지 않으려면 아래 주석을 해제하세요 (Headless 모드)
    # chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    # 2. 트위터 검색 URL 이동 (최신 트윗 탭으로 이동: f=live)
    search_url = f"https://x.com/search?q={keyword}&src=typed_query"
    driver.get(search_url)
    print(f"'{keyword}' 검색 결과 페이지로 이동 중... 로그인이 필요할 수 있습니다.")
    
    # 트위터는 로그인 창이 뜰 수 있으므로 페이지 로딩을 위해 5초 대기
    time.sleep(5) 
    
    tweet_data = []
    seen_texts = set()  # 중복 수집 방지
    
    # 3. 스크롤하며 데이터 수집
    for i in range(scroll_count):
        print(f"스크롤 진행 중... ({i+1}/{scroll_count})")
        
        # 트윗 글을 담고 있는 내부 Article 요소 탐색
        articles = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')
        
        for article in articles:
            try:
                # 트윗 본문 텍스트 추출
                text_element = article.find_element(By.XPATH, './/div[@data-testid="tweetText"]')
                tweet_text = text_element.text.replace('\n', ' ')
                
                # 날짜 및 시간 추출 (time 태그의 datetime 속성)
                date_element = article.find_element(By.XPATH, './/time')
                tweet_date = date_element.get_attribute('datetime')
                
                # 중복되지 않은 데이터만 저장
                if tweet_text not in seen_texts:
                    seen_texts.add(tweet_text)
                    tweet_data.append({
                        '날짜': tweet_date,
                        '내용': tweet_text
                    })
            except Exception:
                # 텍스트가 없거나 이미지 전용 트윗 등 예외 발생 시 패스
                continue
        
        # 아래로 스크롤 내리기
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) # 로딩 대기
        
    driver.quit()
    
    # 4. 데이터프레임 변환 및 CSV 다운로드 저장
    df = pd.DataFrame(tweet_data)
    output_filename = 'son_tweets.csv'
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n🎉 스크래핑 완료! 총 {len(df)}개의 트윗이 '{output_filename}'로 저장(다운로드)되었습니다.")
    
    return df

# 스크립트 실행
if __name__ == "__main__":
    # 키워드와 스크롤 횟수 설정 (원하는 만큼 늘릴 수 있습니다)
    scrape_twitter(keyword="손흥민", scroll_count=5)

'손흥민' 검색 결과 페이지로 이동 중... 로그인이 필요할 수 있습니다.
스크롤 진행 중... (1/5)
스크롤 진행 중... (2/5)
스크롤 진행 중... (3/5)
스크롤 진행 중... (4/5)
스크롤 진행 중... (5/5)

🎉 스크래핑 완료! 총 0개의 트윗이 'son_tweets.csv'로 저장(다운로드)되었습니다.
